In [ ]:
# Imports
from pathlib import Path
import re

import pandas as pd
import numpy as np

In [ ]:
# Load fox and nbc data
fox = pd.read_csv("../data/raw/fox_scraped_all.csv")
nbc = pd.read_csv("../data/raw/nbc_scraped_all.csv")

# --- Fox Categories---
fox['category'] = fox['url'].str.extract(r'foxnews\.com/([^/]+)')

threshold = 5
counts = fox['category'].value_counts()
fox['category'] = fox['category'].replace(counts[counts < threshold].index, 'other')

# --- NBC Categories ---
nbc_cat = nbc['url'].str.extract(r'nbcnews\.com/([^/]+)')[0]
nbc_sub = nbc['url'].str.extract(r'nbcnews\.com/news/([^/]+)')[0]
nbc['category'] = nbc_sub.where(nbc_cat == 'news', other=nbc_cat)

# Fix slug artifacts first
def clean_nbc_category(cat):
    if pd.isna(cat):
        return 'other'
    if len(cat) > 40 or cat.count('-') > 5:
        return 'other'
    return cat

nbc['category'] = nbc['category'].apply(clean_nbc_category)

# Collapse small counts, preserve identity verticals
preserve = {'nbc-out', 'nbcblk', 'latino', 'asian-america'}
counts = nbc['category'].value_counts()
small = set(counts[counts < threshold].index) - preserve
nbc['category'] = nbc['category'].replace(list(small), 'other')

# Combine into one dataframe
news_data = pd.concat([fox, nbc], ignore_index = True)

In [ ]:
# Fox category distribution
print("Number of categories:", fox["category"].nunique())
print(fox['category'].value_counts())

Number of categories: 15
category
politics          615
media             385
lifestyle         203
us                201
entertainment     166
world             110
sports             90
health             73
travel             41
opinion            39
food-drink         35
official-polls     14
tech               12
live-news          10
other               6
Name: count, dtype: int64


In [ ]:
# NBC category distribution
print("Number of categories:", nbc["category"].nunique())
print(nbc['category'].value_counts())

Number of categories: 18
category
politics          679
world             435
select            274
us-news            73
tech               68
latino             38
other              36
investigations     32
asian-america      27
science            27
health             26
nbc-out            19
nbcblk             17
pop-culture        15
weather            12
business           11
sports             10
meet-the-press      6
Name: count, dtype: int64


In [61]:
# Convert datetime_posted to datetime
news_data["datetime_posted"] = pd.to_datetime(news_data["datetime_posted"], format = "mixed", utc = True)

# Basic checks
print(fox.shape, nbc.shape, news_data.shape)

(2000, 8) (1805, 8) (3805, 8)


In [71]:
# Copy news_data
news_df = news_data.copy()

# Get rid of topic column
news_df = news_df.drop(columns = ["topic"])

# One hot encode label column
news_df["is_fox"] = (news_df["label"] == "Fox News").astype(int)

# Check that number of fox and nbc articles match
print("Original vs OHE fox counts:\n\n", news_data["label"].value_counts(), "\n\n", news_df["is_fox"].value_counts())

# Drop label column
news_df = news_df.drop(columns = ["label"])

Original vs OHE fox counts:

 label
Fox News    2000
NBC News    1805
Name: count, dtype: int64 

 is_fox
1    2000
0    1805
Name: count, dtype: int64


In [ ]:
news_df.head() # Quick look

,url,title,subtitle,author,datetime_posted,category,is_fox
0,https://www.foxnews.com/politics/blinken-meets...,"Blinken meets Qatar PM, says Israeli actions a...",Blinken stressed the importance of protecting ...,Timothy H.J. Nerozzi,2023-10-13 18:06:08+00:00,politics,1
1,https://www.foxnews.com/entertainment/bruce-wi...,"Bruce Willis, Demi Moore avoided doing one thi...",Demi Moore and Bruce Willis have been divorced...,Lauryn Overhultz,2024-10-18 19:56:05+00:00,entertainment,1
2,https://www.foxnews.com/media/the-view-co-host...,"'The View' co-host, CNN commentator Ana Navarr...","'It’s just such a mind-blowing moment,' Navarr...",Kristine Parks,2024-08-20 01:00:35+00:00,media,1
3,https://www.foxnews.com/entertainment/emily-bl...,Emily Blunt says her ‘toes curl’ when people t...,Blunt and John Krasinski share two daughters; ...,Lauryn Overhultz,2023-06-09 17:55:28+00:00,entertainment,1
4,https://www.foxnews.com/lifestyle/jack-carrs-e...,Jack Carr recalls Gen. Eisenhower's D-Day memo...,Bestselling author recalls Gen. Eisenhower's s...,Jack Carr,2024-06-06 08:30:24+00:00,lifestyle,1


In [74]:
# Write news_df to csv in data/processed
news_df.to_csv("../data/processed/combined_base_data.csv", index = False)